In [166]:
"""

Hourly Data Assimilation and Spatial Interpolation using Kriging

Part A: Build an hourly index covering the study period 
1. Station data
    1a. Joins station data csvs with the metadata csv to bring in elevation, lat/long associated with each station id 
    1b. Collapse station data to the hourly level by... at each target hour, collect all station observations within the hour and 
    average for that station
    1c. Generally the station csvs contain predictors for temp_air, temp_dew, and rh. For any predictors that were missing 
    before (i.e., NA), calculate them using foundational equations found in model_meteo(). 
    Calculate temp_bulb based on equations found in model_meteo(). 
2. IMERG: 
    2a. Convert wide to long and average half hourly data to the hourly level. 
3. MRoS: 
    3a. There might be multiple observations coming from the same observer within an hour. 
    If that's the case, choose the latter observation that was recorded (i.e., if an observer changed their mind about the phase). 
    Otherwise, floor each MRoS observation datetime_UTC to the starting hour. 
4. At this point, all the data should have lat, lon, datetime_utc (hourly level), predictors. 
    Filter all of them to the lat/long within our DEM AOI. 

Part B: Kriging to Surface
Now that all data should be time synchronized at the hourly level, perform spatial interpolations onto the 1km DEM grid using kriging. 
1. Resample the DEM surface to be 1km to free up some compute time down the road. Reproject from degrees to meters.
2. Kriging to grid: perform kriging interpolation on each predictor to the DEM surface/grid. 
    The predictors we use are 
    a) PLP from the imerg dataset
    b) mros_plp_proxy from the MRoS dataset (rain --> 100, snow --> 0, mix --> 50 % prob to match IMERG PLP format), 
    c) t_air, t_wet, t_dew, rh from station datasets (apply lapse rate -0.0005 K m-1 to these variables, except for RH, which is dimensionless)
    Use projected coordinates, fit variogram models, and perform ordinary kriging with minimum of 3 points

"""

# Dependencies: pandas, numpy, pyarrow, geopandas, shapely, rasterio, rioxarray, xarray,
#               pyproj, scipy, tqdm, pykrige, scikit-gstat

import re
import json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import rasterio as rio
from rasterio.warp import transform_bounds, reproject, Resampling, calculate_default_transform
import xarray as xr
import rioxarray  # noqa: F401 (registers .rio accessor)
from pyproj import CRS, Transformer
from scipy.spatial import cKDTree
from tqdm import tqdm
import pytz
from collections import defaultdict
import math
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import netCDF4

# Kriging-specific imports
from pykrige.ok import OrdinaryKriging
#from pykrige.variogram_models import gaussian, spherical, exponential, linear 
PYKRIGE_AVAILABLE = True
import skgstat as skg
SKGSTAT_AVAILABLE = True



In [167]:
# --------------------------- CONFIG ---------------------------------
BASE_DIR = Path().resolve().parent  # current working dir (folder where you launched jupyter)
print("BASE_DIR:", BASE_DIR)


CONFIG = {
    "wy_start": "2024-10-01T00:00:00Z",
    "wy_end":   "2025-05-31T23:59:59Z",
    "test_start": "2025-05-17T12:00:00Z",   # narrow test window first
    "test_end":   "2025-05-17T15:00:00Z",
    # "test_start": "2024-10-01T00:00:00Z",   # Entire window
    # "test_end":   "2025-05-31T23:59:59Z",

    "station_meta_csv": BASE_DIR / "Data/Stations/station_metadata_20241001_20250531.csv",
    "station_dir": BASE_DIR / "Data/Stations",   # per-station CSVs
    "imerg_dir":   BASE_DIR / "Data/IMERG",      # parquet (wide)
    "mros_parquet": BASE_DIR / "Data/observations/wy25_mros_obs.parquet",

    # Emma: "dem_path": "C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/DEM_AOI_TNM_10m.tif",
    # Zeed:
    "dem_path": r"../DEM_1km.tif",
    "out_dir":  BASE_DIR / "outputs/hourly_pipeline",

    # Kriging-specific parameters
    "min_points": 3,
    "lapse_K_per_m": -0.005,   # constant lapse for temps
    "proj_fallback": "EPSG:3310",  # if DEM is geographic
    
    # Variogram model parameters
    "variogram_model": "spherical",  # Options: 'linear', 'power', 'gaussian', 'spherical', 'exponential'
    "variogram_parameters": {
        "sill": None,      # Will be estimated from data
        "range": None,     # Will be estimated from data  
        "nugget": 0.0,     # Nugget effect
    },
    
    # Kriging parameters
    "enable_plotting": False,  # Set to True to plot variograms (slower)
    "max_points_for_variogram": 100,  # Limit points for variogram fitting to avoid memory issues
    "kriging_method": "ordinary",  # 'ordinary' or 'universal'
    
    # Fallback to IDW if kriging fails
    "fallback_to_idw": False,
    "idw_power": 2.0,
    "k_nearest": 8,
}

out_dir = Path(CONFIG["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

BASE_DIR: C:\Users\zeeda\OneDrive - Desert Research Institute\Desktop\DRI-Keith's project\mros-precipitation-phase-product-prototype


In [168]:
# Load already projected and saved 1km DEM tif

def grid_centers(profile):
    T = profile["transform"]
    xs = T.c + (np.arange(profile["width"]) + 0.5) * T.a
    ys = T.f + (np.arange(profile["height"]) + 0.5) * T.e
    X, Y = np.meshgrid(xs, ys)
    return np.column_stack([X.ravel(), Y.ravel()])

# Emma: with rio.open(r"C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/mros-precipitation-phase-product-prototype/DEM_1km.tif") as src:
# Zeed:
with rio.open(r"../DEM_1km.tif") as src:
    dem1k_profile = src.profile   # metadata
    dem1k_data = src.read(1)      # pixel values

    # Optional extras
    dem_crs = src.crs             # CRS object
    dem_bounds = src.bounds       # bounding box
    dem_transform = src.transform # affine transform

grid_xy = grid_centers(dem1k_profile)
grid_elev = dem1k_data.ravel()
proj_crs = dem1k_profile["crs"]

print(f"DEM 1-km grid: {dem1k_profile['width']} x {dem1k_profile['height']} | "
      f"res ≈ {abs(dem1k_profile['transform'].a)} m")


DEM 1-km grid: 324 x 486 | res ≈ 0.00925926088637072 m


In [169]:
# Load hourly-level stations, IMERG, and MRoS if already performed:

st_hr   = pd.read_parquet(r"../outputs/hourly_pipeline/stations_hourly.parquet")
imerg_hr = pd.read_parquet(r"../outputs/hourly_pipeline/imerg_hourly.parquet")
mros     = pd.read_parquet(r"../outputs/hourly_pipeline/mros_hourly.parquet")


### Checking the results

In [162]:
imerg_hr

,hour_utc,lat,lon,plp
1,2024-10-11 00:00:00+00:00,35.549999,-120.949998,100.0
2,2024-10-11 00:00:00+00:00,35.549999,-120.849998,100.0
3,2024-10-11 00:00:00+00:00,35.549999,-120.749998,100.0
4,2024-10-11 00:00:00+00:00,35.549999,-120.649998,100.0
5,2024-10-11 00:00:00+00:00,35.549999,-120.549998,100.0
...,...,...,...,...
7358124,2025-05-30 23:00:00+00:00,39.949999,-118.449998,100.0
7358125,2025-05-30 23:00:00+00:00,39.949999,-118.349998,100.0
7358126,2025-05-30 23:00:00+00:00,39.949999,-118.249998,100.0
7358127,2025-05-30 23:00:00+00:00,39.949999,-118.149998,100.0


In [150]:
# 1) Make sure the column is tz-aware UTC datetimes (and aligned to the hour)
imerg_hr["hour_utc"] = (
    pd.to_datetime(imerg_hr["hour_utc"], utc=True, errors="coerce")
      .dt.floor("h")
)

# 2) Parse your window bounds as UTC datetimes
start = pd.to_datetime(CONFIG["test_start"], utc=True)
end   = pd.to_datetime(CONFIG["test_end"],   utc=True)

# 3) Filter rows INSIDE the window (inclusive)
inside_imerg_hr = imerg_hr.loc[(imerg_hr["hour_utc"] >= start) & (imerg_hr["hour_utc"] <= end)]


In [151]:
inside_imerg_hr

,hour_utc,lat,lon,plp
6879025,2025-05-17 00:00:00+00:00,35.549999,-120.949998,100.0
6879026,2025-05-17 00:00:00+00:00,35.549999,-120.849998,100.0
6879027,2025-05-17 00:00:00+00:00,35.549999,-120.749998,100.0
6879028,2025-05-17 00:00:00+00:00,35.549999,-120.649998,100.0
6879029,2025-05-17 00:00:00+00:00,35.549999,-120.549998,100.0
...,...,...,...,...
6884692,2025-05-17 03:00:00+00:00,39.949999,-118.449998,100.0
6884693,2025-05-17 03:00:00+00:00,39.949999,-118.349998,100.0
6884694,2025-05-17 03:00:00+00:00,39.949999,-118.249998,100.0
6884695,2025-05-17 03:00:00+00:00,39.949999,-118.149998,100.0


In [152]:
st_hr

,id,hour_utc,temp_air,temp_dew,rh,lat,lon,elev,temp_wet
0,1049,2024-10-01 08:00:00+00:00,5.111111,NaN,NaN,38.68,-119.9600,2443.5816,NaN
1,1049,2024-10-01 09:00:00+00:00,4.722222,NaN,NaN,38.68,-119.9600,2443.5816,NaN
2,1049,2024-10-01 10:00:00+00:00,4.222222,NaN,NaN,38.68,-119.9600,2443.5816,NaN
3,1049,2024-10-01 11:00:00+00:00,3.722222,NaN,NaN,38.68,-119.9600,2443.5816,NaN
4,1049,2024-10-01 12:00:00+00:00,3.500000,NaN,NaN,38.68,-119.9600,2443.5816,NaN
...,...,...,...,...,...,...,...,...,...
692469,YYVC1,2025-05-31 19:00:00+00:00,26.361111,NaN,46.50,37.74,-119.5889,1246.0000,18.597746
692470,YYVC1,2025-05-31 20:00:00+00:00,27.222222,NaN,46.25,37.74,-119.5889,1246.0000,19.282533
692471,YYVC1,2025-05-31 21:00:00+00:00,29.861111,NaN,36.50,37.74,-119.5889,1246.0000,19.643012
692472,YYVC1,2025-05-31 22:00:00+00:00,24.902778,NaN,55.25,37.74,-119.5889,1246.0000,18.713039


In [153]:
# 1) Make sure the column is tz-aware UTC datetimes (and aligned to the hour)
st_hr["hour_utc"] = (
    pd.to_datetime(st_hr["hour_utc"], utc=True, errors="coerce")
      .dt.floor("h")
)

# 2) Parse your window bounds as UTC datetimes
start = pd.to_datetime(CONFIG["test_start"], utc=True)
end   = pd.to_datetime(CONFIG["test_end"],   utc=True)

# 3) Filter rows INSIDE the window (inclusive)
inside_st_hr = st_hr.loc[(st_hr["hour_utc"] >= start) & (st_hr["hour_utc"] <= end)]


In [154]:
inside_st_hr

,id,hour_utc,temp_air,temp_dew,rh,lat,lon,elev,temp_wet
5440,1049,2025-05-17 00:00:00+00:00,18.111111,NaN,NaN,38.68,-119.96,2443.5816,NaN
5441,1049,2025-05-17 01:00:00+00:00,18.000000,NaN,NaN,38.68,-119.96,2443.5816,NaN
5442,1049,2025-05-17 02:00:00+00:00,16.777778,NaN,NaN,38.68,-119.96,2443.5816,NaN
5443,1049,2025-05-17 03:00:00+00:00,14.500000,NaN,NaN,38.68,-119.96,2443.5816,NaN
11248,1050,2025-05-17 00:00:00+00:00,10.722222,NaN,44.0,38.84,-119.89,2608.1736,5.154481
...,...,...,...,...,...,...,...,...,...
426923,846,2025-05-17 03:00:00+00:00,13.777778,NaN,NaN,38.07,-119.23,2865.1200,NaN
432728,848,2025-05-17 00:00:00+00:00,8.222222,NaN,NaN,39.14,-120.22,2055.8760,NaN
432729,848,2025-05-17 01:00:00+00:00,10.111111,NaN,NaN,39.14,-120.22,2055.8760,NaN
432730,848,2025-05-17 02:00:00+00:00,7.388889,NaN,NaN,39.14,-120.22,2055.8760,NaN


In [155]:
mros

,hour_utc,lat,lon,mros_plp_proxy,phase
15,2024-10-06 21:00:00+00:00,39.5129988333333,-119.92934,100.0,rain
16,2024-10-06 21:00:00+00:00,39.5248703161207,-119.870792414262,100.0,rain
9,2024-10-06 23:00:00+00:00,39.5248703161207,-119.870792414262,100.0,rain
12,2024-10-07 01:00:00+00:00,39.3828431931648,-119.777903114124,100.0,rain
2,2024-10-07 02:00:00+00:00,39.4790712,-119.8322994,100.0,rain
...,...,...,...,...,...
31285,2025-05-17 22:00:00+00:00,39.4203380813969,-119.794067616015,100.0,rain
31294,2025-05-17 22:00:00+00:00,39.434450527654,-119.774000999517,100.0,rain
31368,2025-05-18 00:00:00+00:00,39.6412201,-119.8644398,100.0,rain
31353,2025-05-18 01:00:00+00:00,38.8523344549369,-119.997438646991,50.0,mix


In [156]:
# 1) Make sure the column is tz-aware UTC datetimes (and aligned to the hour)
mros["hour_utc"] = (
    pd.to_datetime(mros["hour_utc"], utc=True, errors="coerce")
      .dt.floor("h")
)

# 2) Parse your window bounds as UTC datetimes
start = pd.to_datetime(CONFIG["test_start"], utc=True)
end   = pd.to_datetime(CONFIG["test_end"],   utc=True)

# 3) Filter rows INSIDE the window (inclusive)
inside_mros = mros.loc[(mros["hour_utc"] >= start) & (mros["hour_utc"] <= end)]


In [157]:
inside_mros

,hour_utc,lat,lon,mros_plp_proxy,phase


## Part B: Kriging Interpolation


In [10]:
# -------------------- Old Production Kriging Function ------------------------------------

def build_transformer(src_epsg: str, dst_crs):
    return Transformer.from_crs(src_epsg, dst_crs, always_xy=True)

def kriging_grid_from_points(hour_points: pd.DataFrame,
                            grid_xy: np.ndarray,
                            grid_elev: np.ndarray,
                            proj_crs,
                            min_points=3,
                            value_col="temp_air",
                            station_elev_col="elev",
                            apply_lapse=False, lapse=-0.005,
                            variogram_model="spherical",
                            variogram_params=None,
                            max_points=50,
                            enable_plotting=False,
                            chunk_size=1000):
    """
    Production-ready kriging interpolation from point observations to grid.
    
    This function handles memory management, error recovery, and provides optimal performance.
    
    Parameters:
    -----------
    hour_points : pd.DataFrame
        Point observations with columns [lon, lat, value_col, station_elev_col]
    grid_xy : np.ndarray
        Grid coordinates (N, 2) in projected CRS
    grid_elev : np.ndarray
        Grid elevations (N,)
    proj_crs : CRS
        Projected coordinate reference system
    min_points : int
        Minimum number of points required for interpolation
    value_col : str
        Column name for the variable to interpolate
    station_elev_col : str
        Column name for station elevation
    apply_lapse : bool
        Whether to apply lapse rate correction
    lapse : float
        Lapse rate in K/m
    variogram_model : str
        Variogram model type ('spherical', 'gaussian', 'exponential', 'linear')
    variogram_params : dict
        Variogram parameters (sill, range, nugget). If None or contains None values,
        automatic fitting will be used.
    max_points : int
        Maximum number of points to use for variogram fitting (default: 50)
    enable_plotting : bool
        Whether to plot variogram (slower)
    chunk_size : int
        Size of chunks for processing large grids (default: 1000)
    
    Returns:
    --------
    np.ndarray
        Interpolated values on grid (float32)
    """
    
    # Check if pykrige is available
    if not PYKRIGE_AVAILABLE:
        print("Warning: pykrige not available, falling back to IDW")
        return idw_grid_from_points(hour_points, grid_xy, grid_elev, proj_crs,
                                   CONFIG.get("idw_power", 2.0), 
                                   CONFIG.get("k_nearest", 8), 
                                   min_points, value_col, station_elev_col, 
                                   apply_lapse, lapse)
    
    # Clean input data
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"])
    if pts.empty or pts[value_col].notna().sum() < min_points:
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)
    
    # Limit points for variogram fitting to manage memory (O(n²) complexity)
    if len(pts) > max_points:
        pts_sample = pts.sample(n=max_points, random_state=42)
    else:
        pts_sample = pts.copy()
    
    # Transform station coordinates to projected CRS
    tf = build_transformer("EPSG:4326", proj_crs)
    px, py = tf.transform(pts_sample["lon"].values, pts_sample["lat"].values)
    
    # Apply lapse rate correction if needed
    values = pts_sample[value_col].values.astype(float)
    if apply_lapse and station_elev_col in pts_sample:
        stn_elev = pts_sample[station_elev_col].values.astype(float)
        
        # Calculate reference elevation from grid (handle NaN values)
        if grid_elev is not None and len(grid_elev) > 0:
            valid_elev_mask = ~np.isnan(grid_elev)
            if np.any(valid_elev_mask):
                ref_elev = np.mean(grid_elev[valid_elev_mask])
            else:
                ref_elev = np.mean(stn_elev)
        else:
            ref_elev = np.mean(stn_elev)
        
        # Safety check for None/NaN reference elevation
        if np.isnan(ref_elev) or ref_elev is None:
            ref_elev = 0.0
            
        # Apply lapse rate correction
        values = values + lapse * (ref_elev - stn_elev)
    
    try:
        # Create OrdinaryKriging object with automatic variogram fitting if needed
        if variogram_params is None or any(v is None for v in variogram_params.values() 
                                         if isinstance(variogram_params, dict)):
            OK = OrdinaryKriging(
                px, py, values,
                variogram_model=variogram_model,
                verbose=False,
                enable_plotting=enable_plotting,
                coordinates_type='euclidean'
            )
            print("Kriging with automatic variogram fitting")
        else:
            OK = OrdinaryKriging(
                px, py, values,
                variogram_model=variogram_model,
                variogram_parameters=variogram_params,
                verbose=False,
                enable_plotting=enable_plotting,
                coordinates_type='euclidean'
            )
            print("Kriging with defined variogram parameters")
        
        # Perform kriging interpolation in chunks to manage memory
        n_chunks = (len(grid_xy) + chunk_size - 1) // chunk_size
        z_pred = np.full(len(grid_xy), np.nan, dtype=np.float32)
        
        for i in range(n_chunks):
            start_idx = i * chunk_size
            end_idx = min((i + 1) * chunk_size, len(grid_xy))
            chunk_xy = grid_xy[start_idx:end_idx]
            
            try:
                # Use 'points' mode for individual point predictions
                chunk_pred, chunk_var = OK.execute('points', 
                                                  chunk_xy[:, 0], 
                                                  chunk_xy[:, 1])
                z_pred[start_idx:end_idx] = chunk_pred.astype(np.float32)
            except Exception as e:
                # Fill with NaN for failed chunks
                z_pred[start_idx:end_idx] = np.nan
        
        return z_pred
        
    except Exception as e:
        print(f"Kriging failed: {e}. Falling back to IDW.")
        return idw_grid_from_points(hour_points, grid_xy, grid_elev, proj_crs,
                                   CONFIG.get("idw_power", 2.0), 
                                   CONFIG.get("k_nearest", 8), 
                                   min_points, value_col, station_elev_col, 
                                   apply_lapse, lapse)

print("Production kriging function defined successfully!")


Production kriging function defined successfully!


In [170]:
# -------------------- New Production Kriging Function ------------------------------------

def build_transformer(src_epsg: str, dst_crs):
    return Transformer.from_crs(src_epsg, dst_crs, always_xy=True)

def _coerce_variogram_params(params):
    """
    Coerce variogram params to a form PyKrige accepts (list or dict).
    Returns None if params are invalid or non-finite.
    """
    # FIX 4: Robust handling of variogram_params types (dict/array/tuple/None) so PyKrige accepts them.
    if params is None:
        return None
    if isinstance(params, dict):
        return params
    try:
        arr = np.asarray(params, dtype=float).ravel()
    except Exception:
        return None
    if not np.all(np.isfinite(arr)):
        return None
    return arr.tolist()


def kriging_grid_from_points(hour_points: pd.DataFrame,
                             grid_xy: np.ndarray,
                             grid_elev: np.ndarray,
                             proj_crs,
                             min_points=3,
                             value_col="temp_air",
                             station_elev_col="elev",
                             apply_lapse=False, lapse=-0.005,
                             variogram_model="spherical",
                             variogram_params=None,
                             max_points=50,
                             enable_plotting=False,
                             chunk_size=1000,
                             return_variance: bool = False):  # FIX 3: Flag to optionally return variance
    """
    Production-ready kriging interpolation from point observations to grid.

    Parameters
    ----------
    hour_points : pd.DataFrame
        Columns: ['lon','lat', value_col, station_elev_col]
    grid_xy : np.ndarray
        Grid coordinates (N, 2) in projected CRS
    grid_elev : np.ndarray
        Grid elevations (N,)
    proj_crs : rasterio.crs.CRS or pyproj.CRS
    min_points : int
    value_col : str
    station_elev_col : str
    apply_lapse : bool
    lapse : float
    variogram_model : {'spherical','gaussian','exponential','linear'}
    variogram_params : dict | list | tuple | None
        If None (or dict with any None), auto-fit on sample is used.
    max_points : int
        Number of points for variogram fitting (sample size)
    enable_plotting : bool
    chunk_size : int
    return_variance : bool
        If True, also return variance (same length as z_pred)

    Returns
    -------
    np.ndarray  or  (np.ndarray, np.ndarray)
        Predicted values on grid (float32), and optionally variance (float32).
    """

    # If pykrige is unavailable, fall back to IDW
    if not PYKRIGE_AVAILABLE:
        print("Warning: pykrige not available, falling back to IDW")
        z_pred = idw_grid_from_points(hour_points, grid_xy, grid_elev, proj_crs,
                                      CONFIG.get("idw_power", 2.0),
                                      CONFIG.get("k_nearest", 8),
                                      min_points, value_col, station_elev_col,
                                      apply_lapse, lapse)
        if return_variance:  # FIX 3: When falling back to IDW, return NaN variance to preserve API.
            v_pred = np.full_like(z_pred, np.nan, dtype=np.float32)
            return z_pred, v_pred
        return z_pred

    # Clean input
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"]).reset_index(drop=True)
    if pts.empty or pts[value_col].notna().sum() < min_points:
        if return_variance:  # FIX 3: Maintain (values, variance) return shape when insufficient points.
            nan_arr = np.full(grid_elev.shape, np.nan, dtype=np.float32)
            return nan_arr, nan_arr.copy()
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    # Transform ALL points once; sample by index later
    tf = build_transformer("EPSG:4326", proj_crs)
    px_all, py_all = tf.transform(pts["lon"].values, pts["lat"].values)
    values_all = pts[value_col].values.astype(float)
    # FIX 1: Prepare "ALL points" arrays (px_all, py_all, values_all) for kriging on all points.

    # Choose sample indices for variogram fitting
    if len(pts) > max_points:
        rng = np.random.RandomState(42)
        sample_idx = rng.choice(len(pts), size=max_points, replace=False)
    else:
        sample_idx = np.arange(len(pts), dtype=int)

    # Slice sample arrays
    px_s = px_all[sample_idx]
    py_s = py_all[sample_idx]
    values_s = values_all[sample_idx]
    # FIX 1: Prepare "SAMPLE" arrays (px_s, py_s, values_s) used only for variogram fitting.

    # Lapse detrend (same ref_elev for both sets)
    if apply_lapse and (station_elev_col in pts.columns):
        stn_elev_all = pts[station_elev_col].values.astype(float)

        if grid_elev is not None and np.isfinite(grid_elev).any():
            ref_elev = float(np.nanmean(grid_elev))
        else:
            ref_elev = float(np.nanmean(stn_elev_all))
        if not np.isfinite(ref_elev):
            ref_elev = 0.0

        values_all = values_all + lapse * (ref_elev - stn_elev_all)
        values_s   = values_s   + lapse * (ref_elev - stn_elev_all[sample_idx])
        # FIX 1: Apply the SAME lapse-rate detrend to BOTH sets (all points and sample) before fitting/kriging.
    else:
        ref_elev = 0.0  # harmless placeholder when apply_lapse=False

    # Decide auto-fit vs provided params (robust guard)
    use_auto = (
        variogram_params is None or
        (isinstance(variogram_params, dict) and any(v is None for v in variogram_params.values()))
    )
    # FIX 4: Robust guard to decide auto-fit vs fixed params without calling .values() on non-dicts.

    try:
        # 1) Fit variogram on SAMPLE (auto if requested)
        if use_auto:
            OK_fit = OrdinaryKriging(px_s, py_s, values_s,
                                     variogram_model=variogram_model,
                                     verbose=False,
                                     enable_plotting=enable_plotting,
                                     coordinates_type='euclidean')
            fitted_params = getattr(OK_fit, "variogram_model_parameters",
                             getattr(OK_fit, "variogram_parameters", None))
            fitted_params = _coerce_variogram_params(fitted_params)  # <--- normalize
            print("Kriging: auto-fit variogram on sample.")
            # FIX 1 + FIX 4: Fit variogram on the SAMPLE only; normalize fitted params to acceptable type.
        else:
            fitted_params = _coerce_variogram_params(variogram_params)  # <--- normalize
            print("Kriging: using provided variogram parameters.")
            # FIX 4: Normalize user-provided variogram parameters to list/dict.

        # 2) Build OK with ALL points using fitted/known params
        if fitted_params is None:
            # No valid params -> let PyKrige auto-fit on all points as a fallback
            OK = OrdinaryKriging(px_all, py_all, values_all,
                                 variogram_model=variogram_model,
                                 verbose=False,
                                 enable_plotting=enable_plotting,
                                 coordinates_type='euclidean')
        else:
            OK = OrdinaryKriging(px_all, py_all, values_all,
                                 variogram_model=variogram_model,
                                 variogram_parameters=fitted_params,
                                 verbose=False,
                                 enable_plotting=enable_plotting,
                                 coordinates_type='euclidean')
            print("Build OK with ALL points using fitted/known params")
        # FIX 1: Krige with ALL points (px_all, py_all, values_all) using params fitted on the sample.

        # 3) Predict in chunks
        n = len(grid_xy)
        n_chunks = (n + chunk_size - 1) // chunk_size
        z_pred = np.full(n, np.nan, dtype=np.float32)
        v_pred = np.full(n, np.nan, dtype=np.float32) if return_variance else None  # FIX 3: collect variance if requested

        for i in range(n_chunks):
            s = i * chunk_size
            e = min((i + 1) * chunk_size, n)
            chunk_xy = grid_xy[s:e]
            try:
                chunk_z, chunk_v = OK.execute('points', chunk_xy[:, 0], chunk_xy[:, 1])
                z_pred[s:e] = np.asarray(chunk_z, dtype=np.float32)
                if return_variance:
                    v_pred[s:e] = np.asarray(chunk_v, dtype=np.float32)  # FIX 3: aggregate kriging variance
            except Exception:
                # leave NaNs for this chunk
                pass

        # 4) Re-apply lapse per grid cell (undo detrend to ref_elev)
        if apply_lapse and grid_elev is not None:
            z_pred = z_pred + lapse * (grid_elev - ref_elev)
            # variance unchanged by deterministic adjustment
            # FIX 2: After kriging (which was at ref_elev), restore cell-specific elevation using lapse.

        print("Kriging: re-applied lapse to all grid cells")
        return (z_pred, v_pred) if return_variance else z_pred  # FIX 3: Return (values, variance) when requested

    except Exception as e:
        print(f"Kriging failed: {e}. Falling back to IDW.")
        z_pred = idw_grid_from_points(hour_points, grid_xy, grid_elev, proj_crs,
                                      CONFIG.get("idw_power", 2.0),
                                      CONFIG.get("k_nearest", 8),
                                      min_points, value_col, station_elev_col,
                                      apply_lapse, lapse)
        if return_variance:  # FIX 3: Keep return signature; variance is NaN under IDW fallback.
            v_pred = np.full_like(z_pred, np.nan, dtype=np.float32)
            return z_pred, v_pred
        return z_pred


In [171]:
# -------------------- IDW Functions ------------------------------------
def build_transformer(src_epsg: str, dst_crs):
    return Transformer.from_crs(src_epsg, dst_crs, always_xy=True)

def idw_grid_from_points(hour_points: pd.DataFrame,
                         grid_xy: np.ndarray,
                         grid_elev: np.ndarray,
                         proj_crs,
                         idw_power=2.0, k=8, min_points=3,
                         value_col="temp_air",
                         station_elev_col="elev",
                         apply_lapse=False, lapse=-0.005):
    """Fallback IDW function for when kriging fails"""
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"])
    if pts.empty or pts[value_col].notna().sum() < min_points:
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    # transform station coords into the same projection
    tf = build_transformer("EPSG:4326", proj_crs)
    px, py = tf.transform(pts["lon"].values, pts["lat"].values)
    P = np.column_stack([px, py])

    values = pts[value_col].values.astype(float)
    stn_elev = pts[station_elev_col].values.astype(float) if station_elev_col in pts else np.zeros_like(values)

    # nearest neighbor search, for each grid cell, finds up to k nearest stations
    tree = cKDTree(P)
    dists, idxs = tree.query(grid_xy, k=min(k, len(P)))
    if dists.ndim == 1:
        dists = dists[:, None]
        idxs  = idxs[:,  None]

    # get neighbor station values for each grid cell, apply lapse rate on select parameters to account for temp change with elevation
    v_neighbors = values[idxs]
    if apply_lapse:
        zc = grid_elev[:, None]
        zj = stn_elev[idxs]
        v_neighbors = v_neighbors + lapse * (zc - zj)

    # compute weights
    with np.errstate(divide="ignore"):
        w = 1.0 / np.power(dists, idw_power)
    w[np.isinf(w)] = 1e12
    w[~np.isfinite(w)] = 0.0
    # normalize weightsm ensure weights sum to 1 per cell
    w_sum = w.sum(axis=1, keepdims=True)
    w_norm = np.divide(w, w_sum, out=np.zeros_like(w), where=w_sum > 0)

    valid_counts = np.sum(w > 0, axis=1)
    # weighted sum (weighted average of neighbor values)
    grid_vals = np.sum(w_norm * v_neighbors, axis=1)
    grid_vals[valid_counts < min_points] = np.nan
    return grid_vals.astype(np.float32)

In [172]:
# -------------------- Hourly Assimilation ------------------------------------

def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
                         freq="H", tz="UTC")

def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

out_dir = Path(CONFIG["out_dir"]); out_dir.mkdir(parents=True, exist_ok=True)
hours = hourly_index(CONFIG["test_start"], CONFIG["test_end"])

variables = [
    ("temp_air",       "station", True),
    ("temp_dew",       "station", True),
    ("temp_wet",       "station", True),
    ("rh",             "station", False),
    ("mros_plp_proxy", "mros",    False),
    ("plp",            "imerg",   False),
]

# Build coords from the DEM 1-km profile (use rasterio.xy to avoid any drift)
from rasterio.transform import xy as rio_xy
H, W = dem1k_profile["height"], dem1k_profile["width"]
T = dem1k_profile["transform"]

rows = np.arange(H)
cols = np.arange(W)
# centers from affine; one row vector for x, one col vector for y
x_centers = np.array([rio_xy(T, 0.5, c + 0.5, offset="center")[0] for c in cols])
y_centers = np.array([rio_xy(T, r + 0.5, 0.5, offset="center")[1] for r in rows])

coords = {
    "time": hours,
    "y": y_centers,
    "x": x_centers,
}
data_vars = {
    name: np.full((len(hours), H, W), np.nan, dtype=np.float32)
    for (name, _, _) in variables
}

def summarize_points(st_t, mros_t, imerg_t, min_points):
    msg = []
    ns = st_t.dropna(subset=["temp_air","temp_dew","rh"]).shape[0]
    msg.append(f"stations rows: {ns}")
    msg.append(f"mros rows: {mros_t.dropna(subset=['mros_plp_proxy']).shape[0]}")
    msg.append(f"imerg rows: {imerg_t.dropna(subset=['plp']).shape[0]}")
    msg.append("vars_ok: " + ", ".join([
        f"Ta={int(st_t['temp_air'].notna().sum()>=min_points)}",
        f"Td={int(st_t['temp_dew'].notna().sum()>=min_points)}",
        f"Tw={int(('temp_wet' in st_t) and (st_t['temp_wet'].notna().sum()>=min_points))}",
        f"RH={int(st_t['rh'].notna().sum()>=min_points)}",
        f"MRoS={int(mros_t['mros_plp_proxy'].notna().sum()>=min_points)}",
        f"PLP={int(imerg_t['plp'].notna().sum()>=min_points)}"
    ]))
    return " | ".join(msg)

for ti, t in enumerate(tqdm(hours, desc="Hourly surfaces", ncols=88)):
    st_t   = st_hr[st_hr["hour_utc"] == t]
    mros_t = mros[mros["hour_utc"] == t]
    imerg_t= imerg_hr[imerg_hr["hour_utc"] == t]

    print(f"[{print_time(t)}] {summarize_points(st_t, mros_t, imerg_t, CONFIG['min_points'])}")

    for name, src, use_lapse in tqdm(variables, desc=f"  vars {print_time(t)}", leave=False, ncols=88):
        if src == "station":
            if st_t.empty or st_t[name].notna().sum() < CONFIG["min_points"]:
                continue
            pts = st_t[["lon","lat","elev", name]]
            vals = kriging_grid_from_points(
                pts, grid_xy, grid_elev, proj_crs,
                min_points=CONFIG["min_points"], value_col=name,
                station_elev_col="elev",
                apply_lapse=use_lapse, lapse=CONFIG["lapse_K_per_m"],
                variogram_model=CONFIG["variogram_model"],
                variogram_params=CONFIG["variogram_parameters"],
                max_points=CONFIG["max_points_for_variogram"],
                enable_plotting=CONFIG["enable_plotting"]
            )
            data_vars[name][ti, :, :] = vals.reshape(H, W)

        elif src == "mros":
            if mros_t.empty or mros_t["mros_plp_proxy"].notna().sum() < CONFIG["min_points"]:
                continue
            pts = mros_t.rename(columns={"mros_plp_proxy":"val"})[["lon","lat","val"]].assign(elev=0.0)
            vals = kriging_grid_from_points(
                pts, grid_xy, grid_elev, proj_crs,
                min_points=CONFIG["min_points"], value_col="val",
                station_elev_col="elev", apply_lapse=False,
                variogram_model=CONFIG["variogram_model"],
                variogram_params=CONFIG["variogram_parameters"],
                max_points=CONFIG["max_points_for_variogram"],
                enable_plotting=CONFIG["enable_plotting"]
            )
            data_vars[name][ti, :, :] = vals.reshape(H, W)
            
        elif src == "imerg":
            if imerg_t.empty or imerg_t["plp"].notna().sum() < CONFIG["min_points"]:
                continue
            pts = imerg_t.rename(columns={"plp":"val"})[["lon","lat","val"]].assign(elev=0.0)
            vals = kriging_grid_from_points(
                pts, grid_xy, grid_elev, proj_crs,
                min_points=CONFIG["min_points"], value_col="val",
                station_elev_col="elev", apply_lapse=False,
                variogram_model=CONFIG["variogram_model"],
                variogram_params=CONFIG["variogram_parameters"],
                max_points=CONFIG["max_points_for_variogram"],
                enable_plotting=CONFIG["enable_plotting"]
            )
            assert vals.size == H * W, f"Kriging returned {vals.size} cells but grid is {H*W}"
            data_vars[name][ti, :, :] = vals.reshape(H, W)

# assemble dataset
ds = xr.Dataset(
    {
        **{k: xr.DataArray(v, coords=coords, dims=("time","y","x"))
           for k, v in data_vars.items()},
        "elev": xr.DataArray(
            dem1k_data.astype(np.float32),
            coords={"y": y_centers, "x": x_centers},
            dims=("y","x"),
        ),
    },
    attrs={
        "title": "Hourly predictor stacks on 1-km grid using Kriging",
        "interpolation_method": "Ordinary Kriging",
        "lapse_K_per_m": CONFIG["lapse_K_per_m"],
        "variogram_model": CONFIG["variogram_model"],
        "variogram_parameters": str(CONFIG["variogram_parameters"]),
        "min_points": CONFIG["min_points"],
        "max_points_for_variogram": CONFIG["max_points_for_variogram"],
        "fallback_to_idw": CONFIG["fallback_to_idw"],
    }
)

# Coordinate metadata (meters)
ds["x"].attrs.update({
    "units": "m",
    "standard_name": "projection_x_coordinate",
    "long_name": "x coordinate of projection",
})
ds["y"].attrs.update({
    "units": "m",
    "standard_name": "projection_y_coordinate",
    "long_name": "y coordinate of projection",
})

# Make geospatial + CF-compliant (creates a 'spatial_ref' variable)
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds = ds.rio.write_crs(dem1k_profile["crs"], grid_mapping_name="spatial_ref")
ds = ds.rio.write_transform(dem1k_profile["transform"])

# Ensure each data variable points to the grid mapping
for v in ds.data_vars:
    ds[v].attrs["grid_mapping"] = "spatial_ref"

# Add a GDAL-style GeoTransform (helps some viewers)
A = dem1k_profile["transform"]
ds.attrs["GeoTransform"] = f"{A.c} {A.a} {A.b} {A.f} {A.d} {A.e}"

Hourly surfaces:   0%|                                            | 0/4 [00:00<?, ?it/s]

[2025-05-17 12:00Z] stations rows: 26 | mros rows: 0 | imerg rows: 1350 | vars_ok: Ta=1, Td=1, Tw=1, RH=1, MRoS=0, PLP=1


Kriging: auto-fit variogram on sample.
Build OK with ALL points using fitted/known params


Kriging: re-applied lapse to all grid cells
Kriging: auto-fit variogram on sample.
Build OK with ALL points using fitted/known params


Kriging: re-applied lapse to all grid cells
Kriging: auto-fit variogram on sample.
Build OK with ALL points using fitted/known params


...

Kriging: re-applied lapse to all grid cells
Kriging: auto-fit variogram on sample.
Build OK with ALL points using fitted/known params


Kriging: re-applied lapse to all grid cells
Kriging: auto-fit variogram on sample.
Build OK with ALL points using fitted/known params


Kriging: re-applied lapse to all grid cells
Kriging: auto-fit variogram on sample.
Build OK with ALL points using fitted/known params


Hourly surfaces: 100%|███████████████████████████████████| 4/4 [09:33<00:00, 143.47s/it]

Kriging: re-applied lapse to all grid cells


In [173]:
# -------------------- Fix NetCDF4 Boolean Attribute Issue ------------------------------------

# Fix the boolean attribute that's causing the error because NetCDF4 doesn't support boolean attributes
ds.attrs["fallback_to_idw"] = str(ds.attrs["fallback_to_idw"])

print("Fixed boolean attribute for NetCDF4 compatibility")
print(f"fallback_to_idw is now: {ds.attrs['fallback_to_idw']} (type: {type(ds.attrs['fallback_to_idw'])})")


Fixed boolean attribute for NetCDF4 compatibility
fallback_to_idw is now: False (type: <class 'str'>)


In [174]:
# -------------------- Save NetCDFs ------------------------------------
out_dir = Path(CONFIG["out_dir"]); out_dir.mkdir(parents=True, exist_ok=True)
out_nc = out_dir / "test3_3hours_hourly_predictors_1km_kriging.nc"

# Ensure time is tz-naive
if hasattr(ds.indexes["time"], "tz") and ds.indexes["time"].tz is not None:
    ds = ds.assign_coords(time=ds.indexes["time"].tz_localize(None))

# Reassert spatial metadata (idempotent & safe)
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds = ds.rio.write_crs(dem1k_profile["crs"])               # <- use DEM CRS object
ds = ds.rio.write_transform(dem1k_profile["transform"])   # <- use DEM affine


# CF link each data var to the grid mapping (spatial_ref)
for v in ds.data_vars:
    ds[v].attrs.setdefault("grid_mapping", "spatial_ref")

# Build encoding per variable (match chunks to dims!)
def _chunks_for(da):
    # cap chunk sizes to something sane
    if da.dims == ("time", "y", "x"):
        return (min(24, da.sizes["time"]),
                min(256, da.sizes["y"]),
                min(256, da.sizes["x"]))
    if da.dims == ("y", "x"):
        return (min(256, da.sizes["y"]),
                min(256, da.sizes["x"]))
    return None  # scalar or unusual dims

encoding = {}
for name, da in ds.data_vars.items():
    ch = _chunks_for(da)
    encoding[name] = ({"zlib": True, "complevel": 4, "chunksizes": ch}
                      if ch is not None else {"zlib": True, "complevel": 4} if da.ndim > 0 else {})

# (coords like x/y/time generally don’t need custom encoding)
ds.to_netcdf(out_nc, engine="netcdf4", encoding=encoding)
print(f"Wrote {out_nc} using netCDF4 (compressed).")


Wrote C:\Users\zeeda\OneDrive - Desert Research Institute\Desktop\DRI-Keith's project\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\test3_3hours_hourly_predictors_1km_kriging.nc using netCDF4 (compressed).


In [15]:
# # Check netcdf
# from netCDF4 import Dataset

# nc = Dataset(out_nc, mode="r")

# # Dimensions
# print("\nDimensions:")
# for name, dim in nc.dimensions.items():
#     print(f"  {name}: {len(dim)}")

# # Variables
# print("\nVariables:")
# for name, var in nc.variables.items():
#     print(f"  {name}: shape={var.shape}, dtype={var.dtype}, attrs={ {k: v for k, v in var.__dict__.items()} }")

# # Check the data
# out_nc = Path(CONFIG["out_dir"]) / "hourly_predictors_1km.nc"
# ds = xr.open_dataset(out_nc)

# # Print a quick summary again
# print(ds)

# # Inspect first few timesteps for one variable (e.g. temp_air)
# print("\nFirst 2 timesteps of temp_air, temp_wet, temp_dew, rh, mros_plp_proxy, plp: at 5x5 corner:")
# print(ds["temp_air"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["temp_wet"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["temp_dew"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["rh"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["mros_plp_proxy"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["plp"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)

# ds.close()


In [175]:
# -------------------- Quick Plotting ------------------------------------
from pyproj import CRS

def quicklook_hour(ds, t, st_t, mros_t, out_png, vars_to_show=(
    "plp","mros_plp_proxy","temp_air","temp_dew","temp_wet","rh")):

    if t not in ds.time.values:
        print(f"No time {t} in dataset for quicklook.")
        return

    # Extent for imshow
    xvals = ds["x"].values
    yvals = ds["y"].values
    extent = [xvals.min(), xvals.max(), yvals.min(), yvals.max()]

    keep = [v for v in vars_to_show if v in ds.data_vars]
    if not keep:
        print("No matching variables to plot.")
        return
    n = len(keep) 
    ncols = 3
    nrows = int(np.ceil(len(keep)/ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows), squeeze=False)
    fig.suptitle(f"Quicklook @ {print_time(t)}", fontsize=14)

    # Make sure we have a CRS 
    if getattr(ds.rio, "crs", None):
        target_crs = ds.rio.crs
    else:
        target_crs = CRS.from_user_input(CONFIG["proj_fallback"])

    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
    print("Dataset CRS:", ds.rio.crs)

    # Project obs points
    st_x = st_y = mo_x = mo_y = []
    if len(st_t):
        st_x, st_y = tf.transform(st_t["lon"].values,  st_t["lat"].values)
    if len(mros_t):
        mo_x, mo_y = tf.transform(mros_t["lon"].values, mros_t["lat"].values)

    ti = int(np.where(ds.time.values == np.datetime64(t))[0][0])

    for i, var in enumerate(keep):
        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        # color scale
        if var in ("plp", "mros_plp_proxy"):
            im = ax.imshow(arr, origin="lower", extent=extent, aspect="equal", vmin=0, vmax=100)
        elif var == "rh":
            im = ax.imshow(arr, origin="lower", extent=extent, aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="lower", extent=extent, aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("x"); ax.set_ylabel("y")

        # scatter obs
        if len(st_x):
            ax.scatter(st_x, st_y, s=15, c="white", edgecolor="k",
                       marker="o", linewidths=0.5, label="Stations")
        if len(mo_x):
            ax.scatter(mo_x, mo_y, s=25, c="red", edgecolor="k",
                       marker="^", linewidths=0.6, label="MRoS")

        # repeat legend on each subplot
        ax.legend(loc="upper right", frameon=True, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    for j in range(n, nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(out_png, dpi=200); plt.close(fig)
    print(f"Saved quicklook: {out_png}")

# sample a few hours
quick_dir = Path(CONFIG["out_dir"]) / "maps"; quick_dir.mkdir(parents=True, exist_ok=True)
sample_hours = pd.to_datetime(ds.time.values)[::max(1, len(ds.time)//6)]
for t in sample_hours:
    t_utc = pd.to_datetime(t).tz_localize("UTC").floor("H")
    st_t = st_hr[st_hr["hour_utc"].dt.floor("H") == t_utc]
    mros_t = mros[mros["hour_utc"].dt.floor("H") == t_utc]
    print(f"[{t_utc}] Stations: {len(st_t)}, MRoS: {len(mros_t)}")

    quicklook_hour(ds, t, st_t, mros_t, out_png=quick_dir / f"test3_Kriging_quick_{print_time(t).replace(':','-')}.png")


[2025-05-17 12:00:00+00:00] Stations: 56, MRoS: 0
Dataset CRS: EPSG:4326
Saved quicklook: C:\Users\zeeda\OneDrive - Desert Research Institute\Desktop\DRI-Keith's project\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_Kriging_quick_2025-05-17 12-00Z.png
[2025-05-17 13:00:00+00:00] Stations: 56, MRoS: 0
Dataset CRS: EPSG:4326
Saved quicklook: C:\Users\zeeda\OneDrive - Desert Research Institute\Desktop\DRI-Keith's project\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_Kriging_quick_2025-05-17 13-00Z.png
[2025-05-17 14:00:00+00:00] Stations: 56, MRoS: 0
Dataset CRS: EPSG:4326
Saved quicklook: C:\Users\zeeda\OneDrive - Desert Research Institute\Desktop\DRI-Keith's project\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_Kriging_quick_2025-05-17 14-00Z.png
[2025-05-17 15:00:00+00:00] Stations: 56, MRoS: 0
Dataset CRS: EPSG:4326
Saved quicklook: C:\Users\zeeda\OneDrive - Desert Research Institute\Deskto